# Ingestão no S3

In [1]:
#Bibliotecas
import zipfile
import io
import boto3
import os
import pandas as pd
import pyarrow
import openpyxl
from dotenv import load_dotenv

In [2]:
# carregar .env
load_dotenv()

True

In [3]:
#Variável Máquina local
caminho_w = os.getenv("CAMINHO_W")

#Variável arquivos
pasta_arquivos_zip = f"{caminho_w}data\\zip"
pasta_local = f"{caminho_w}data\\bases"

#Variaveis do bucket
bucket_name = os.getenv("BUCKET_NAME")
prefix_s3 = "bronze/pnad_covid"

## Destino S3

No Bucket deve conter as seguintes pastas : 
- bronze
- silver
- gold

In [4]:
#Configurações da Sessão AWS
session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    aws_session_token=os.getenv("AWS_SESSION_TOKEN"),
    region_name=os.getenv("AWS_DEFAULT_REGION")
)

s3 = session.client("s3")
print(s3.list_buckets())

{'ResponseMetadata': {'RequestId': 'FWB0VJ4FXFKF3KQE', 'HostId': 'mvuWLSSNJ3bmv+YWSHKL7+uWturv/pJ08czUBuC9X7qcF+56xIBFr1c3cqNDjT+w2EGhK69r4OLWMhfkA/DZzQLIgrh4STWH', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'mvuWLSSNJ3bmv+YWSHKL7+uWturv/pJ08czUBuC9X7qcF+56xIBFr1c3cqNDjT+w2EGhK69r4OLWMhfkA/DZzQLIgrh4STWH', 'x-amz-request-id': 'FWB0VJ4FXFKF3KQE', 'date': 'Sun, 26 Apr 2026 18:37:16 GMT', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'aws-glue-assets-404604958697-us-east-1', 'CreationDate': datetime.datetime(2026, 2, 11, 19, 35, 38, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::aws-glue-assets-404604958697-us-east-1'}, {'Name': 'datalake-404604958697', 'CreationDate': datetime.datetime(2026, 2, 27, 13, 58, 29, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::datalake-404604958697'}, {'Name': 'lab-404604958697', 'CreationDate': datetime.datetime(2026, 2, 11, 18, 47, 54, tzinfo=tzutc()), 'BucketArn': 'a

In [5]:
s3_client = boto3.client('s3')

# Leitura Bases


In [6]:
for arquivo in os.listdir(pasta_arquivos_zip):
    if arquivo.endswith(".zip"):
        caminho_zip = os.path.join(pasta_arquivos_zip, arquivo)

        # Nome base
        nome_base = arquivo.replace(".zip", "")

        # extrair ano e mes do nome
        mes = nome_base.split("_")[2][:2]
        ano = nome_base.split("_")[2][2:]

        # abrir zip em memória
        with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:

            # pegar arquivos dentro do zip
            for nome_arquivo in zip_ref.namelist():

                if nome_arquivo.endswith(".csv"):

                    # ler CSV direto do zip (sem salvar no disco)
                    with zip_ref.open(nome_arquivo) as file:
                        df = pd.read_csv(file, sep=",")

                    # converter para parquet em memória
                    buffer = io.BytesIO()
                    df.to_parquet(buffer, index=False)
                    buffer.seek(0)

                    # caminho no S3
                    caminho_s3 = f"{prefix_s3}/ano={ano}/mes={mes}/dados.parquet"

                    # upload
                    s3.upload_fileobj(buffer, bucket_name, caminho_s3)

                    print(f"Enviado: {caminho_s3}")

print("Processo concluído!")

Enviado: bronze/pnad_covid/ano=2020/mes=05/dados.parquet
Enviado: bronze/pnad_covid/ano=2020/mes=11/dados.parquet
Enviado: bronze/pnad_covid/ano=2020/mes=10/dados.parquet
Enviado: bronze/pnad_covid/ano=2020/mes=09/dados.parquet
Enviado: bronze/pnad_covid/ano=2020/mes=08/dados.parquet
Enviado: bronze/pnad_covid/ano=2020/mes=07/dados.parquet
Enviado: bronze/pnad_covid/ano=2020/mes=06/dados.parquet
Processo concluído!


# Subindo Dicionario

In [8]:
caminho_dicionario = f"{caminho_w}data\\bases\\Dicionario_PNAD_COVID_052020_20220621.xls"

# Ler Excel
df = pd.read_excel(caminho_dicionario, header=1)
# Corrigir células mescladas
df = df.ffill()

# -----------------------------
# Linhas para remover
# -----------------------------
linhas_remover = [0,2,1, 94, 137, 254, 474, 503]
df_linhas_removidas = df.loc[linhas_remover].copy()

# -----------------------------
# Remover do dataframe principal
# -----------------------------
df = df.drop(linhas_remover)

# Selecionar apenas colunas úteis do dicionário
df = df[[
    "Código\nda\nvariável",
    "Unnamed: 3",
    "Categorias",
    "Unnamed: 5"
]]
# Renomear colunas
df.columns = [
    "cod_variavel",
    "descricao_variavel",
    "cod_categoria",
    "descricao_categoria"
]

#Limpeza dos dados
df["cod_variavel"] = df["cod_variavel"].astype(str)
df["descricao_categoria"] = df["descricao_categoria"].astype(str)
df["descricao_variavel"] = df["descricao_variavel"].astype(str)

# Ajustar "Não aplicável"
mask_na = df["descricao_categoria"].str.contains(
    "Não aplic",
    case=False,
    na=False
)
df.loc[mask_na, "cod_categoria"] = 99
df["cod_categoria"] = df["cod_categoria"].astype(str)

# Criar ID único
df["id_categoria"] = (
    df["cod_variavel"] + "_" + df["cod_categoria"] + "_" + df["descricao_categoria"]
)

# Remover duplicados
df = df.drop_duplicates(subset=["id_categoria"])
# Remover linhas sem categoria
df = df.dropna(subset=["cod_categoria"])

# Criar buffer parquet em memória
buffer = io.BytesIO()
df.to_parquet(buffer,index=False,engine="pyarrow",compression="snappy")
buffer.seek(0)

# Upload para Bronze
s3.upload_fileobj(
    buffer,
    bucket_name,
    "bronze/dicionario_pnad/dicionario_pnad.parquet"
)

print("Dicionário normalizado e enviado com sucesso!")

Dicionário normalizado e enviado com sucesso!


In [9]:
#Mostrar linhas removidas
df_linhas_removidas

,Tamanho,Código\nda\nvariável,Quesito,Unnamed: 3,Categorias,Unnamed: 5
0,NaN,NaN,nº,Descrição,Tipo,Descrição
2,4,Ano,nº,Ano de referência,Tipo,Descrição
1,Parte 1 - Identificação e Controle,NaN,nº,Descrição,Tipo,Descrição
94,Parte A - Características gerais dos moradores,posest,nº,Domínios de projeção,6 dígitos e 8 casas decimais,As 2 primeiras posições representam o código d...
137,Parte B - COVID19 - Todos os moradores,A005,A5,Escolaridade,8,"Pós-graduação, mestrado ou doutorado"
254,Parte C - Características de trabalho das pess...,B007,B7,"Tem algum plano de saúde médico, seja particul...",9,Ignorado
474,Parte D - Rendimento de outras fontes dos mora...,C017A,C17a,"Embora você não tenha procurado trabalho, gost...",2,Não aplicável
503,Parte Suplementar 01 - Características da habi...,D0073,D1,Somatório dos valores recebidos,valor em reais,Não aplicável
